### Colorer les provinces de Madagascar

Prolem: How to colour Madagascar with only three colours?

`Modelization`

X: {Tana, Majunga, Tamatave, Fianarantsoa, Diego, Tulear}

D: {[Red, Green, Blue],[Red, Green, Blue],[Red, Green, Blue], [Red, Green, Blue], [Red, Green, Blue], [Red, Green, Blue]}

C: C_t != C_m, ....

In [1]:
prov = ["Tana", "Majunga", "Tamatave", "Fianarantsoa", "Diego", "Tulear"]
col = ["Red", "Green", "Blue"]
contrainte = [
("Tana","Majunga"),
("Tana","Tamatave"),
("Tana","Fianarantsoa"),
("Tana","Tulear"),
("Diego","Majunga"),
("Diego","Tamatave"),
("Tamatave","Majunga"),
("Tamatave","Fianarantsoa"),
("Majunga","Tulear"),
("Fianarantsoa","Tulear")
]

print(prov[0])

Tana


# Generate & Test

In [2]:
n = len(prov)

assign = [0] * n

print(assign)

[0, 0, 0, 0, 0, 0]


In [5]:
def is_valid(solution, constraints):
    for a, b in constraints:
        if solution[a] == solution[b]:
            return False
    return True
    
while True:
    # indices -> couleurs
    sol = {}
    for i in range(n):
        sol[prov[i]] = col[assign[i]]
        #print(prov[i], assign[i])

    # test contrainte
    if is_valid(sol, contrainte):
        print("Solution trouvée:", sol)
        break

    # generate
    i = n-1
    while i >=0:
        assign[i] += 1
        if assign[i] < len(col):
            break
        else:
            assign[i] = 0
            i -= 1
    if i < 0:
        print("Pas de solution")
        break
        

Solution trouvée: {'Tana': 'Red', 'Majunga': 'Green', 'Tamatave': 'Blue', 'Fianarantsoa': 'Green', 'Diego': 'Red', 'Tulear': 'Blue'}


# BACKTRACKING

Backtracking fait :

- choisir une province
- donner une couleur
- tester immédiatement
- si conflit → revenir en arrière


Donc on coupe l’arbre très tôt.

In [13]:
solution = {}

In [14]:
def consistence(provi, color):
    for a,b in contrainte:
        if a == provi:
            if b in solution and solution[b] == color:
                return False
        if b == provi:
            if a in solution and solution[a] == color:
                return False
    return True                

In [23]:
def backtracking(index):
    if index == len(prov):
        return True

    provi = prov[index]

    for color in col:
        if consistence(provi, color):
            solution[provi] = color
            if backtracking(index + 1):
                return True
            del solution[provi]
    return False

In [24]:
backtracking(0)

print(solution)


{'Tana': 'Red', 'Majunga': 'Green', 'Tamatave': 'Blue', 'Fianarantsoa': 'Green', 'Diego': 'Red', 'Tulear': 'Blue'}


# Forward Checking

réduire le domaine des voisins

Idée :

Quand on assigne une couleur à une province, on supprime cette couleur du domaine des voisins.

quand on assigne une couleur à une variable

→ on supprime cette couleur du domaine des voisins

→ si un voisin n’a plus de couleur possible

→ on revient en arrière immédiatement


In [25]:
domain = {p: col.copy() for p in prov}
solution = {}

print(domain)

{'Tana': ['Red', 'Green', 'Blue'], 'Majunga': ['Red', 'Green', 'Blue'], 'Tamatave': ['Red', 'Green', 'Blue'], 'Fianarantsoa': ['Red', 'Green', 'Blue'], 'Diego': ['Red', 'Green', 'Blue'], 'Tulear': ['Red', 'Green', 'Blue']}


In [26]:
def voisins(p):
    v = []
    for a,b in contrainte:
        if a == p:
            v.append(b)
        elif b == p:
            v.append(a)
    return v


In [27]:
def forward_check(province, color):

    removed = []

    for v in voisins(province):

        if v not in solution and color in domain[v]:

            domain[v].remove(color)
            removed.append((v, color))

            if len(domain[v]) == 0:
                return False, removed

    return True, removed


In [28]:
def restore(removed):
    for v,c in removed:
        domain[v].append(c)


In [29]:
def backtrack(index):

    if index == len(prov):
        return True

    p = prov[index]

    for color in domain[p]:

        solution[p] = color

        ok, removed = forward_check(p, color)

        if ok:
            if backtrack(index+1):
                return True

        restore(removed)
        del solution[p]

    return False


In [30]:
backtrack(0)

print(solution)


{'Tana': 'Red', 'Majunga': 'Green', 'Tamatave': 'Blue', 'Fianarantsoa': 'Green', 'Diego': 'Red', 'Tulear': 'Blue'}
